# Multi-Class Grounding Failure Analysis
Analyze where the model succeeds and fails in multi-class object grounding.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict, Counter
from pathlib import Path
from PIL import Image
from scipy.optimize import linear_sum_assignment

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# Load predictions - change path for finetuned
PRED_FILE = 'results/baseline_multiclass/predictions.json'
with open(PRED_FILE) as f:
    predictions = json.load(f)
print(f'Loaded {len(predictions)} predictions from {PRED_FILE}')

In [ ]:
# ── Helpers ──

def iou(b1, b2):
    x1, y1 = max(b1[0],b2[0]), max(b1[1],b2[1])
    x2, y2 = min(b1[2],b2[2]), min(b1[3],b2[3])
    if x2<x1 or y2<y1: return 0.0
    inter = (x2-x1)*(y2-y1)
    a1 = (b1[2]-b1[0])*(b1[3]-b1[1])
    a2 = (b2[2]-b2[0])*(b2[3]-b2[1])
    return inter/(a1+a2-inter) if (a1+a2-inter)>0 else 0.0

def box_area(b):
    return (b[2]-b[0]) * (b[3]-b[1])

def hungarian_match(gt_boxes, gt_labels, pred_boxes, pred_labels, threshold=0.5):
    """Per-category Hungarian matching. Returns matched, missed_gt, hallucinated_pred."""
    all_cats = set(gt_labels) | set(pred_labels)
    matched = []  # (gt_idx, pred_idx, iou, label)
    missed_gt = []  # (gt_box, label)
    hallucinated = []  # (pred_box, label)
    
    for cat in all_cats:
        cat_gt = [(i,b) for i,(b,l) in enumerate(zip(gt_boxes, gt_labels)) if l == cat]
        cat_pred = [(i,b) for i,(b,l) in enumerate(zip(pred_boxes, pred_labels)) if l == cat]
        
        if not cat_gt:
            hallucinated.extend([(b, cat) for _,b in cat_pred])
            continue
        if not cat_pred:
            missed_gt.extend([(b, cat) for _,b in cat_gt])
            continue
        
        cost = np.zeros((len(cat_gt), len(cat_pred)))
        for i, (_,gb) in enumerate(cat_gt):
            for j, (_,pb) in enumerate(cat_pred):
                cost[i,j] = -iou(gb, pb)
        
        row_ind, col_ind = linear_sum_assignment(cost)
        matched_gt = set()
        matched_pred = set()
        for r, c in zip(row_ind, col_ind):
            m_iou = -cost[r,c]
            matched.append((cat_gt[r][1], cat_pred[c][1], m_iou, cat))
            matched_gt.add(r)
            matched_pred.add(c)
        
        for i, (_,b) in enumerate(cat_gt):
            if i not in matched_gt:
                missed_gt.append((b, cat))
        for i, (_,b) in enumerate(cat_pred):
            if i not in matched_pred:
                hallucinated.append((b, cat))
    
    return matched, missed_gt, hallucinated

In [ ]:
# ── Run matching on all predictions ──

results = []
for pred in predictions:
    gt_boxes = pred['ground_truth_boxes']
    gt_labels = pred['ground_truth_labels']
    pred_boxes = pred['predicted_boxes']
    pred_labels = pred['predicted_labels']
    
    matched, missed, halluc = hungarian_match(gt_boxes, gt_labels, pred_boxes, pred_labels)
    results.append({
        'pred': pred,
        'matched': matched,
        'missed_gt': missed,
        'hallucinated': halluc,
        'n_gt': len(gt_boxes),
        'n_pred': len(pred_boxes),
    })

total_gt = sum(r['n_gt'] for r in results)
total_pred = sum(r['n_pred'] for r in results)
total_matched = sum(len(r['matched']) for r in results)
total_missed = sum(len(r['missed_gt']) for r in results)
total_halluc = sum(len(r['hallucinated']) for r in results)
matched_ious = [m[2] for r in results for m in r['matched']]

print(f'GT boxes: {total_gt}, Predicted: {total_pred}')
print(f'Matched: {total_matched}, Missed GT: {total_missed}, Hallucinated: {total_halluc}')
print(f'Mean matched IoU: {np.mean(matched_ious):.4f}')
print(f'Recall@0.5: {sum(1 for i in matched_ious if i>=0.5)/total_gt:.4f}')
print(f'Precision@0.5: {sum(1 for i in matched_ious if i>=0.5)/total_pred:.4f}')

## 1. Per-Category Performance

In [ ]:
cat_stats = defaultdict(lambda: {'gt':0, 'matched':0, 'correct':0, 'missed':0, 'halluc':0, 'ious':[]})

for r in results:
    for _, _, m_iou, cat in r['matched']:
        cat_stats[cat]['matched'] += 1
        cat_stats[cat]['ious'].append(m_iou)
        if m_iou >= 0.5:
            cat_stats[cat]['correct'] += 1
    for _, cat in r['missed_gt']:
        cat_stats[cat]['missed'] += 1
    for _, cat in r['hallucinated']:
        cat_stats[cat]['halluc'] += 1

for cat in cat_stats:
    cat_stats[cat]['gt'] = cat_stats[cat]['matched'] + cat_stats[cat]['missed']

# Sort by number of GT instances
sorted_cats = sorted(cat_stats.items(), key=lambda x: x[1]['gt'], reverse=True)

# Top 20 most common categories
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

top_cats = sorted_cats[:20]
names = [c[0] for c in top_cats]
recalls = [c[1]['correct']/c[1]['gt'] if c[1]['gt']>0 else 0 for c in top_cats]
mean_ious = [np.mean(c[1]['ious']) if c[1]['ious'] else 0 for c in top_cats]

axes[0].barh(names, recalls, color='#2196F3')
axes[0].set_xlabel('Recall@0.5')
axes[0].set_title('Top 20 Categories by Count - Recall')
axes[0].invert_yaxis()

# Worst 20 categories (min 3 instances)
worst_cats = sorted([c for c in sorted_cats if c[1]['gt']>=3], key=lambda x: x[1]['correct']/x[1]['gt'] if x[1]['gt']>0 else 0)[:20]
names_w = [c[0] for c in worst_cats]
recalls_w = [c[1]['correct']/c[1]['gt'] if c[1]['gt']>0 else 0 for c in worst_cats]

axes[1].barh(names_w, recalls_w, color='#F44336')
axes[1].set_xlabel('Recall@0.5')
axes[1].set_title('Worst 20 Categories (min 3 instances)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 2. Instance Count Analysis

In [ ]:
# How does performance change with number of objects per image?
count_bins = defaultdict(lambda: {'ious': [], 'missed': 0, 'halluc': 0, 'gt': 0})

for r in results:
    n = r['n_gt']
    bucket = '1' if n==1 else '2-5' if n<=5 else '6-10' if n<=10 else '11-20' if n<=20 else '20+'
    count_bins[bucket]['gt'] += r['n_gt']
    count_bins[bucket]['missed'] += len(r['missed_gt'])
    count_bins[bucket]['halluc'] += len(r['hallucinated'])
    count_bins[bucket]['ious'].extend([m[2] for m in r['matched']])

order = ['1', '2-5', '6-10', '11-20', '20+']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric, title in zip(axes, 
    [lambda b: np.mean(b['ious']) if b['ious'] else 0,
     lambda b: sum(1 for i in b['ious'] if i>=0.5)/b['gt'] if b['gt']>0 else 0,
     lambda b: b['missed']/b['gt'] if b['gt']>0 else 0],
    ['Mean IoU', 'Recall@0.5', 'Miss Rate']):
    vals = [metric(count_bins[k]) for k in order if k in count_bins]
    keys = [k for k in order if k in count_bins]
    ax.bar(keys, vals, color='#4CAF50')
    ax.set_xlabel('Objects per image')
    ax.set_title(title)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 3. Object Size Analysis

In [ ]:
# Performance by GT box size (in normalized 0-1000 coords)
size_data = {'small': [], 'medium': [], 'large': []}
missed_sizes = []

for r in results:
    for gt_box, pred_box, m_iou, cat in r['matched']:
        area = box_area(gt_box)
        bucket = 'small' if area < 10000 else 'medium' if area < 100000 else 'large'
        size_data[bucket].append(m_iou)
    for box, cat in r['missed_gt']:
        missed_sizes.append(box_area(box))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IoU by size
labels = ['small\n(<10K)', 'medium\n(10K-100K)', 'large\n(>100K)']
keys = ['small', 'medium', 'large']
mean_ious = [np.mean(size_data[k]) if size_data[k] else 0 for k in keys]
counts = [len(size_data[k]) for k in keys]
axes[0].bar(labels, mean_ious, color=['#FF9800', '#2196F3', '#4CAF50'])
for i, (v, c) in enumerate(zip(mean_ious, counts)):
    axes[0].text(i, v+0.02, f'n={c}', ha='center', fontsize=9)
axes[0].set_ylabel('Mean IoU')
axes[0].set_title('IoU by Object Size')
axes[0].set_ylim(0, 1)

# Missed object size distribution
axes[1].hist(missed_sizes, bins=50, color='#F44336', alpha=0.7)
axes[1].set_xlabel('Box area (normalized coords)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Missed GT Object Sizes (n={len(missed_sizes)})')

plt.tight_layout()
plt.show()

## 4. Cross-Class Confusion

In [ ]:
# Which categories does the model hallucinate most?
halluc_cats = Counter()
missed_cats = Counter()

for r in results:
    for _, cat in r['hallucinated']:
        halluc_cats[cat] += 1
    for _, cat in r['missed_gt']:
        missed_cats[cat] += 1

# Predicted labels not in GT (wrong class)
wrong_class = Counter()
for pred in predictions:
    gt_set = set(pred['ground_truth_labels'])
    for label in pred['predicted_labels']:
        if label not in gt_set:
            wrong_class[label] += 1

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, data, title, color in zip(axes,
    [halluc_cats.most_common(15), missed_cats.most_common(15), wrong_class.most_common(15)],
    ['Most Hallucinated', 'Most Missed', 'Wrong Class Predictions'],
    ['#FF5722', '#F44336', '#9C27B0']):
    if data:
        names, counts = zip(*data)
        ax.barh(list(names), list(counts), color=color)
        ax.invert_yaxis()
    ax.set_title(title)

plt.tight_layout()
plt.show()

## 5. Prediction Count Analysis

In [ ]:
# Does the model predict too few or too many objects?
gt_counts = [r['n_gt'] for r in results]
pred_counts = [r['n_pred'] for r in results]
diffs = [p - g for g, p in zip(gt_counts, pred_counts)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(gt_counts, pred_counts, alpha=0.3, s=10)
max_val = max(max(gt_counts), max(pred_counts))
axes[0].plot([0, max_val], [0, max_val], 'r--', label='Perfect')
axes[0].set_xlabel('GT count')
axes[0].set_ylabel('Predicted count')
axes[0].set_title('Predicted vs GT Object Count')
axes[0].legend()

axes[1].hist(diffs, bins=range(min(diffs)-1, max(diffs)+2), color='#607D8B', alpha=0.7)
axes[1].axvline(0, color='r', linestyle='--')
axes[1].set_xlabel('Predicted - GT count')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Count Difference (mean={np.mean(diffs):.1f})')

plt.tight_layout()
plt.show()

## 6. Visual Examples

In [ ]:
def visualize_prediction(result, ax=None):
    pred = result['pred']
    img = Image.open(pred['image']).convert('RGB')
    w, h = img.size
    
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(img)
    
    # Draw GT boxes (green dashed)
    for box, label in zip(pred['ground_truth_boxes'], pred['ground_truth_labels']):
        x1, y1, x2, y2 = [c/1000 for c in box]
        rect = patches.Rectangle((x1*w, y1*h), (x2-x1)*w, (y2-y1)*h,
            lw=2, ec='lime', fc='none', ls='--')
        ax.add_patch(rect)
        ax.text(x1*w, y1*h-5, f'GT:{label}', color='lime', fontsize=7, 
                bbox=dict(boxstyle='round,pad=0.1', fc='black', alpha=0.5))
    
    # Draw predicted boxes (red solid)
    for box, label in zip(pred['predicted_boxes'], pred['predicted_labels']):
        x1, y1, x2, y2 = [c/1000 for c in box]
        rect = patches.Rectangle((x1*w, y1*h), (x2-x1)*w, (y2-y1)*h,
            lw=2, ec='red', fc='none')
        ax.add_patch(rect)
        ax.text(x2*w, y2*h+10, f'P:{label}', color='red', fontsize=7,
                bbox=dict(boxstyle='round,pad=0.1', fc='black', alpha=0.5))
    
    n_matched = len(result['matched'])
    n_missed = len(result['missed_gt'])
    n_halluc = len(result['hallucinated'])
    ax.set_title(f"GT:{result['n_gt']} Pred:{result['n_pred']} | "
                 f"Matched:{n_matched} Missed:{n_missed} Halluc:{n_halluc}", fontsize=10)
    ax.axis('off')

In [ ]:
# Worst failures (most missed objects)
worst = sorted(results, key=lambda r: len(r['missed_gt']), reverse=True)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Worst Failures (most missed objects)', fontsize=14)
for ax, r in zip(axes.flat, worst[:6]):
    visualize_prediction(r, ax)
plt.tight_layout()
plt.show()

In [ ]:
# Best successes (high recall, many objects)
best = sorted([r for r in results if r['n_gt'] >= 5], 
              key=lambda r: len(r['missed_gt'])/r['n_gt'])

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Best Successes (high recall, 5+ objects)', fontsize=14)
for ax, r in zip(axes.flat, best[:6]):
    visualize_prediction(r, ax)
plt.tight_layout()
plt.show()

In [ ]:
# Most hallucinations
most_halluc = sorted(results, key=lambda r: len(r['hallucinated']), reverse=True)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Most Hallucinations (false positive boxes)', fontsize=14)
for ax, r in zip(axes.flat, most_halluc[:6]):
    visualize_prediction(r, ax)
plt.tight_layout()
plt.show()